# Train model

In [1]:
import numpy as np
import os
from pathlib import Path

from lapsim.logging.logger import NewLogger
import webdataset
from lapsim.preprocessor.encoder import decode

from lapsim.evals.evaluate import evaluate
from lapsim.normalisation import TransformNormalisation
from lapsim.render import RenderItem, plot_full

BATCH_SIZE = 1024
EPOCHS = 300
FORESIGHT = 120
SAMPLING = 4
NORMAL_SPACING = 10
CHECKPOINT_EVERY = 25

NORMALISATION_BOUNDS_PATH = f"bounds-new.json"

TRAIN_DATASET = "/Users/belle/Developer/MlLapSim/dataset/spliced-again-10/lapsim-train-{00..04}.tar"
VALIDATION_DATASET = "/Users/belle/Developer/MlLapSim/dataset/spliced-again-10/lapsim-validation-00.tar"
TEST_DATASET = "/Users/belle/Developer/MlLapSim/dataset/spliced-again-10/lapsim-test-00.tar"
REAL_DATASET = "/Users/belle/Developer/MlLapSim/dataset/spliced-again-10/lapsim-real-00.tar"

TENSOR_CACHE_DIR = fr"/run/media/belle/Development/Tensors-120f-{NORMAL_SPACING}m"

training_dataset = webdataset.WebDataset(TRAIN_DATASET).map(decode)
validation_dataset = webdataset.WebDataset(VALIDATION_DATASET).map(decode)
test_dataset = webdataset.WebDataset(TEST_DATASET).map(decode)
real_dataset = webdataset.WebDataset(REAL_DATASET).map(decode)


/Users/belle/Developer/MlLapSim/venv/lib/python3.14/site-packages/webdataset/compat.py:379: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn("WebDataset(shardshuffle=...) is None; set explicitly to False or a number")


In [2]:
bounds = TransformNormalisation()
bounds.transform.method = "flat-window"
bounds.transform.foresight = FORESIGHT
bounds.transform.sampling = SAMPLING

if os.path.exists(NORMALISATION_BOUNDS_PATH):
    bounds = TransformNormalisation.load(NORMALISATION_BOUNDS_PATH)
    print("Existing bounds loaded.")

else:
    print("No bounds file found, calculating new ones.")
    for record in training_dataset:
        bounds.extend(record)

    bounds.save(NORMALISATION_BOUNDS_PATH)
    print("Finished calculating bounds.")

# todo must set learning rate
# todo must set the neurons in the model and then start training
# todo must fix issue with the encoding

bounds.transform.single_sample = True


Existing bounds loaded.


In [3]:
from torch.optim import NAdam
from torch.nn import HuberLoss
from lapsim.models.lapsim import LapSimModel
import torch


ls_model = LapSimModel(weights_path=None, bounds=bounds)
print("Params", ls_model.total_params)


loss = HuberLoss()
optimiser = NAdam(ls_model.model.parameters())

def tensor(x):
    return torch.tensor(x, dtype=torch.float32).to(ls_model.device)


Params 547418


In [4]:
from torch.utils.data import DataLoader
import random


training_dataloader = DataLoader(
    training_dataset\
        .shuffle(0)
        .map(bounds.normalise)\
        .map(bounds.transform)\
        .batched(BATCH_SIZE),
    batch_size=None,          
    # num_workers=4,     
)
validation_dataloader = DataLoader(
    validation_dataset\
        .shuffle(0)
        .map(bounds.normalise)\
        .map(bounds.transform)\
        .batched(BATCH_SIZE),
    batch_size=None,          
    # num_workers=4,    
)

best_model_perf = None

new_logger = NewLogger("lapsim", run_name="test")

for epoch in range(EPOCHS):
    ls_model.model.train()
    # todo move the tensorisation to the dataloader
    for i, (x, vehicles, positions, velocities) in enumerate(training_dataloader):
        optimiser.zero_grad()

        pred_pos, pred_vel = ls_model.model(tensor(np.squeeze(x)), tensor(np.squeeze(vehicles)))

        pos_loss = loss(pred_pos, tensor(np.squeeze(positions)))
        vel_loss = loss(pred_vel, tensor(np.squeeze(velocities)))
        total_loss = (2 * pos_loss + vel_loss) / 3
        total_loss.backward()
        optimiser.step()

        new_logger.log_training_data(
            epoch,
            batch=i,
            position_loss=pos_loss.item(),
            velocity_loss=vel_loss.item(),
        )

    # Validation
    ls_model.model.eval()
    with torch.no_grad():
        for i, (x, vehicles, positions, velocities) in enumerate(validation_dataloader):
            val_pred_pos, val_pred_vel = ls_model.model(tensor(np.squeeze(x)), tensor(np.squeeze(vehicles)))
            pos_loss = loss(val_pred_pos, tensor(np.squeeze(positions)))
            vel_loss = loss(val_pred_vel, tensor(np.squeeze(velocities)))
    
            new_logger.log_validation_data(
                epoch,
                batch=i,
                position_loss=pos_loss.item(),
                velocity_loss=vel_loss.item(),
            )

    current_val_loss = np.mean(new_logger.val_pos_loss)
    best_validation_reached = best_model_perf is None or current_val_loss < best_model_perf

    new_logger.flush(epoch, learning_rate=-1)

    if best_validation_reached:
        torch.save(ls_model.model.state_dict(), "ls3-best-val.pt")
        torch.save(optimiser.state_dict(), "ls3-best-val-optim.pt")
        best_model_perf = current_val_loss

    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        bounds.transform.single_sample = False
        # Predict on the real dataset
        requests = list(real_dataset)
        predictions = ls_model.predict(requests)
        pairs = list(zip(requests, predictions))
        evaluations = evaluate(pairs)

        for i in range(3, 8):
            plot_full(
                tracks=[
                    RenderItem(
                        track=pairs[i][0],
                        label="Target",
                        color="green"
                    ),
                    RenderItem(
                        track=pairs[i][1],
                        label="Predicted",
                        color="red"
                    ),
                ],
                title="..."
            )

        # Run evals
        new_logger.log_checkpoint(
            epoch,
            ls_model.model,
            (x[0], vehicles[0]),
            evaluations
        )
        bounds.transform.single_sample = True


/var/folders/db/n3n4mm7d47qfqglfm7x1nd_80000gn/T/ipykernel_71096/3274489560.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(x, dtype=torch.float32).to(ls_model.device)


Epoch: 1, loss/train/position: 0.049036, loss/train/velocity: 0.008233, loss/validation/position: 0.039995, loss/validation/velocity: 0.005847, system/learning_rate: -1.000000
Epoch: 2, loss/train/position: 0.031295, loss/train/velocity: 0.003787, loss/validation/position: 0.027276, loss/validation/velocity: 0.002628, system/learning_rate: -1.000000
Epoch: 3, loss/train/position: 0.025875, loss/train/velocity: 0.002901, loss/validation/position: 0.026880, loss/validation/velocity: 0.002451, system/learning_rate: -1.000000
Epoch: 4, loss/train/position: 0.022882, loss/train/velocity: 0.002586, loss/validation/position: 0.031685, loss/validation/velocity: 0.002724, system/learning_rate: -1.000000
Epoch: 5, loss/train/position: 0.020549, loss/train/velocity: 0.002200, loss/validation/position: 0.019600, loss/validation/velocity: 0.001527, system/learning_rate: -1.000000
Epoch: 6, loss/train/position: 0.018992, loss/train/velocity: 0.001816, loss/validation/position: 0.018063, loss/validat

ReadError: ('empty file', <_io.BufferedReader name='/Users/belle/Developer/MlLapSim/dataset/spliced-again-10/lapsim-real-00.tar'>, '/Users/belle/Developer/MlLapSim/dataset/spliced-again-10/lapsim-real-00.tar')

In [ ]:
bounds.transform.single_sample = False

requests = list(real_dataset)
predictions = ls_model.predict(requests)
pairs = list(zip(requests, predictions))
evaluations = evaluate(pairs)

bounds.transform.single_sample = True